# Hugging Face 的文字嵌入模型

In [16]:
# 模組匯入
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# 初始化文字嵌入模型，這裡使用 BAAI 提供的中文向量模型 bge-large-zh-v1.5
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5")

# 定義要轉換成向量的文字
text = "床前明月光，疑是地上霜"

# 使用模型將文字轉換為向量（embedding）
query_result = embeddings.embed_query(text)

# 印出向量的長度（維度），用以確認向量大小
print("向量長度:", len(query_result))

# 印出向量的前 3 個數值，快速查看向量內容
print("向量內容:", query_result[:3])

向量長度: 1024
向量內容: [0.03762822970747948, -0.008928835391998291, -0.02356129139661789]


# 向量相似度

In [20]:
news_titles = [
    "冷氣團來襲！全台氣溫驟降，北部低溫下探15度",
    "午後雷陣雨頻繁　氣象署提醒山區慎防坍方",
    "秋颱逼近！中央氣象署發布海上颱風警報",
    "第三季GDP成長優於預期　政府上調全年經濟展望",
    "股市連三漲　外資回流帶動電子與金融雙主流",
    "AI晶片需求爆發　台灣半導體廠迎來新一波投資潮",
    "新款折疊手機亮相　引爆消費市場搶購熱潮",
    "歐盟宣布新綠能政策　2030年前逐步淘汰燃煤發電",
    "巴黎奧運倒數百日　中華代表隊備戰進入最後衝刺階段"
]

In [25]:
# 一次將所有新聞標題轉換成向量
news_vectors = embeddings.embed_documents(news_titles)

In [24]:
import numpy as np

# 定義一個函式，用來計算兩個向量之間的餘弦相似度
def cosine_similarity(vec1, vec2):
    # 計算兩個向量的點積（dot product）
    dot = np.dot(vec1, vec2)
    
    # 使用向量的L2範數（歐氏長度）計算分母
    # np.linalg.norm(vec) 會回傳向量的長度
    # 餘弦相似度公式：cos(θ) = (A · B) / (||A|| * ||B||)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


In [36]:
similarity = cosine_similarity(news_vectors[0], news_vectors[0])
print("Cosine Similarity:", similarity)

Cosine Similarity: 1.0


In [35]:
similarity = cosine_similarity(news_vectors[0], news_vectors[1])
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.40663084098067087


In [37]:
similarity = cosine_similarity(news_vectors[0], news_vectors[8])
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.297657555816521


In [38]:
# 將問題轉成向量
question_vector = embeddings.embed_query("明天會下雨嗎？")

# 計算每個新聞標題與問題的相似度
similarities = []
for title, vector in zip(news_titles, news_vectors):
    sim = cosine_similarity(question_vector, vector)
    similarities.append((title, sim))

# 依相似度由高到低排序
similarities.sort(key=lambda x: x[1], reverse=True)

# 輸出結果
for title, score in similarities:
    print(f"{title} -> 相似度: {score:.4f}")

午後雷陣雨頻繁　氣象署提醒山區慎防坍方 -> 相似度: 0.3920
冷氣團來襲！全台氣溫驟降，北部低溫下探15度 -> 相似度: 0.3130
秋颱逼近！中央氣象署發布海上颱風警報 -> 相似度: 0.3010
巴黎奧運倒數百日　中華代表隊備戰進入最後衝刺階段 -> 相似度: 0.2427
新款折疊手機亮相　引爆消費市場搶購熱潮 -> 相似度: 0.1585
股市連三漲　外資回流帶動電子與金融雙主流 -> 相似度: 0.1508
歐盟宣布新綠能政策　2030年前逐步淘汰燃煤發電 -> 相似度: 0.1495
第三季GDP成長優於預期　政府上調全年經濟展望 -> 相似度: 0.1482
AI晶片需求爆發　台灣半導體廠迎來新一波投資潮 -> 相似度: 0.1302


# LM-Studio 的文字嵌入模型

In [44]:
# 模組匯入
from langchain_openai import OpenAIEmbeddings

# 初始化文字嵌入模型，這裡使用 LM Studio 本地服務提供的中文向量模型
embeddings = OpenAIEmbeddings(
    model="text-embedding-bge-large-zh-v1.5",  # 指定要使用的文字嵌入模型名稱
    openai_api_key="not-needed", # LM Studio 不需金鑰
    openai_api_base="http://127.0.0.1:1234/v1" # LM Studio 本地服務的 API 位址
)
# embeddings.embed_query("床前明月光，疑是地上霜") # 測試連接(這裡會失敗)

In [45]:
# 匯入 LangChain 的 Embeddings 基底類別
from langchain_core.embeddings import Embeddings
# 匯入 OpenAI 客戶端，用於呼叫 LM Studio 的 API
from openai import OpenAI

# 自訂 LM Studio 文字嵌入類別，繼承自 LangChain 的 Embeddings
class LmStudioEmbeddings(Embeddings):
    def __init__(self, model_name, url):
        """
        初始化 LM Studio Embeddings
        :param model_name: 要使用的嵌入模型名稱
        :param url: LM Studio 本地或遠端 API 的位址
        """
        self.model_name = model_name
        self.url = url
        # 建立 OpenAI 客戶端，連接 LM Studio 的 API
        self.client = OpenAI(base_url=url, api_key="lm-studio")

    def embed_query(self, text: str):
        """
        將單筆文字轉換為向量
        :param text: 要嵌入的文字
        :return: 向量 (list of float)
        """
        response = self.client.embeddings.create(
            input=text,      # 傳入單筆文字
            model=self.model_name  # 指定使用的模型
        )
        # 回傳第一筆 embedding
        return response.data[0].embedding

    def embed_documents(self, texts: list[str]):
        """
        將多筆文字轉換為向量
        :param texts: 要嵌入的文字列表
        :return: 向量列表，每個元素對應一筆文字
        """
        response = self.client.embeddings.create(
            input=texts,     # 傳入多筆文字
            model=self.model_name  # 指定使用的模型
        )
        # 回傳每筆文字的 embedding
        return [x.embedding for x in response.data]


In [49]:
embeddings = LmStudioEmbeddings(model_name="text-embedding-bge-large-zh-v1.5", url="http://127.0.0.1:1234/v1")
query_result = embeddings.embed_query("床前明月光，疑是地上霜")
print("向量長度:", len(query_result))
print("向量內容:", query_result[:3])

向量長度: 1024
向量內容: [0.04917880892753601, -0.020576421171426773, -0.024843133985996246]
